Richard Davies · **Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# London: mapping millions of prices

London is the biggest part of the menu data: almost half of the whole dataset. You have **every price at London's cafes, pubs and bakeries**: 1.8 million prices from 10,000 venues, 2012 to 2026, in one file, `london/menu_london_cafes_pubs_bakeries.parquet` (22 MB). (It has venues with at least 20 prices. London's restaurants and takeaways are in the 5% sample.)

Read in full, the table takes about 500 MB of memory. So we read only the columns we need.

The LSE Growth Lab collected **3.3 million photos of menus**, posted with Google Maps reviews across 25 UK cities, and used **large language models** (AI that can read images) to pull out the prices: **12.8 million prices** from 80,000 cafes, pubs and restaurants, from 2011 to 2026. It's part of the AutoCPI project, which uses web data to measure prices in more detail than official statistics can: in June 2026, the ONS collected just 115 hospitality prices for the whole of Wales.

In this notebook:
1. **Load** eight columns, and keep 2024–26
2. **Every venue as a point:** too many to see
3. **Condense into a grid:** count venues in squares of about 1 km
4. **The price of a latte, square by square,** and why small squares need care

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites.
# Run this before importing Altair. (If you've already imported it, use Runtime → Restart session, then run again.)
# On your own computer (see our local Python guide), this just checks you have the latest version.
%pip install altair --upgrade --quiet

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()     # Altair stops at 5,000 rows by default: our tables are bigger

DATA = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week08/data/"
MAPS = "https://raw.githubusercontent.com/jhellingsdata/map-data/refs/heads/main/gbr/"

SOURCE = "Source: LSE Growth Lab, AutoCPI (menu photos)"     # for chart subtitles

<br><br>

---

<br>

## 1. Load

Eight columns. This takes a few seconds: it's 22 MB to download.

In [ ]:
columns = ["venue_id", "vtype", "itl1", "canon", "price", "lat", "lng", "date"]
london = pd.read_parquet(DATA + "london/menu_london_cafes_pubs_bakeries.parquet", columns=columns)
london["year"] = pd.to_datetime(london["date"]).dt.year

print(f"{len(london):,} prices")
print(f"{london.memory_usage(deep=True).sum() / 1e6:.0f} MB in memory")

<br>

Keep recent prices (2024 to 2026). The London collection area also reaches into the counties around it, so keep the venues inside the London region (`itl1`):

In [ ]:
recent = london[(london["year"] >= 2024) & (london["itl1"] == "London")]
print(f"{len(recent):,} prices in London, 2024–26")

<br><br>

---

<br>

## 2. Every venue as a point

One row per venue: `drop_duplicates` keeps the first row for each `venue_id`:

In [ ]:
venues = recent.drop_duplicates("venue_id")[["venue_id", "vtype", "lat", "lng"]]
print(f"{len(venues):,} venues")

<br>

The base map is the 33 London boroughs, from map-data's Greater London folder. Each venue is a tiny, see-through dot (`opacity`), so crowded places look darker:

In [ ]:
boroughs = alt.Data(url=MAPS + "sa/msa022_greater_london/LAD.json",
                    format=alt.DataFormat(type="topojson", feature="geog"))

borough_layer = alt.Chart(boroughs).mark_geoshape(fill="#f2f2f2", stroke="#999999", strokeWidth=0.5)

dots = alt.Chart(venues).mark_circle(size=3, opacity=0.15, color="#c8102e").encode(
    longitude=alt.Longitude("lng:Q"),
    latitude=alt.Latitude("lat:Q"),
)

(borough_layer + dots).project("mercator").properties(
    width=500, height=420,
    title=alt.Title("London's cafes, pubs and bakeries, one dot each", subtitle=SOURCE))

<br>

You can see the high streets. But in the centre the dots pile on top of each other, so 50 venues and 500 look the same. And the chart carries every point: saved, it's about 600 KB.

<br><br>

---

<br>

## 3. Condense into a grid

Put each venue in a square about 1 km across, then count the venues in each square. **Floor division** (`//`) does the binning: `lat // 0.01 * 0.01` snaps each latitude down to a multiple of 0.01°, the bottom edge of its square. A degree of longitude is shorter at London's latitude, so the longitude step is 0.016°, to make the squares about square:

In [ ]:
LAT_STEP, LNG_STEP = 0.01, 0.016

venues["cell_lat"] = venues["lat"] // LAT_STEP * LAT_STEP
venues["cell_lng"] = venues["lng"] // LNG_STEP * LNG_STEP

grid = venues.groupby(["cell_lat", "cell_lng"]).size().reset_index(name="venues")
grid["lat2"] = grid["cell_lat"] + LAT_STEP          # the top and right edges of each square
grid["lng2"] = grid["cell_lng"] + LNG_STEP
print(f"{len(venues):,} venues in {len(grid):,} squares")
grid.sort_values("venues", ascending=False).head()

<br>

Each square is a `rect` with two corners: `longitude` and `latitude` for one, `longitude2` and `latitude2` for the other. A **log** colour scale stops the busiest squares (hundreds of venues) from washing out the rest:

In [ ]:
squares = alt.Chart(grid).mark_rect(opacity=0.9).encode(
    longitude=alt.Longitude("cell_lng:Q"),
    latitude=alt.Latitude("cell_lat:Q"),
    longitude2=alt.Longitude2("lng2:Q"),
    latitude2=alt.Latitude2("lat2:Q"),
    color=alt.Color("venues:Q").title("Venues").scale(type="log", scheme="reds"),
    tooltip=[
        alt.Tooltip("venues:Q").format(",").title("Venues"),
    ],
)

(borough_layer + squares).project("mercator").properties(
    width=500, height=420,
    title=alt.Title("Where London's cafes, pubs and bakeries are", subtitle="Venues in each square of about 1 km. " + SOURCE))

<br>

About 7,000 points became about 900 squares, and now the pattern shows: the West End and the City, then high streets running out from the centre.

<br><br>

---

<br>

## 4. The price of a latte, square by square

The same grid, but now for prices: every latte price in each square, and its median. Keep the prices (not one row per venue), and count both the prices and the cafes in each square:

In [ ]:
lattes = recent[recent["canon"] == "latte"].copy()
lattes["cell_lat"] = lattes["lat"] // LAT_STEP * LAT_STEP
lattes["cell_lng"] = lattes["lng"] // LNG_STEP * LNG_STEP

by_cell = lattes.groupby(["cell_lat", "cell_lng"]).agg(
    median=("price", "median"),
    prices=("price", "count"),
    cafes=("venue_id", "nunique"),
).reset_index()
by_cell.head()

<br>

`.agg()` with names: `median=("price", "median")` means "a column called `median`, the median of `price`". It's a neat way to get several summaries, of different columns, in one go.

### Small squares need care

A square with one or two cafes shows **those cafes'** prices, not the area's. That can mislead, and with private data it would disclose a single business (the lecture's *primary disclosure*). So keep squares with at least 5 cafes and 20 prices:

In [ ]:
by_cell = by_cell[(by_cell["cafes"] >= 5) & (by_cell["prices"] >= 20)]
by_cell["lat2"] = by_cell["cell_lat"] + LAT_STEP
by_cell["lng2"] = by_cell["cell_lng"] + LNG_STEP
print(f"{len(by_cell)} squares")

prices = alt.Chart(by_cell).mark_rect(opacity=0.95).encode(
    longitude=alt.Longitude("cell_lng:Q"),
    latitude=alt.Latitude("cell_lat:Q"),
    longitude2=alt.Longitude2("lng2:Q"),
    latitude2=alt.Latitude2("lat2:Q"),
    color=alt.Color("median:Q").title("£").scale(scheme="browns", domain=[3.25, 5], clamp=True),
    tooltip=[
        alt.Tooltip("median:Q").format(".2f").title("Median latte (£)"),
        alt.Tooltip("cafes:Q").format(",").title("Cafes"),
        alt.Tooltip("prices:Q").format(",").title("Prices"),
    ],
)

chart = (borough_layer + prices).project("mercator").properties(
    width=500, height=420,
    title=alt.Title("The price of a latte across London, 2024–26",
                    subtitle=["Median price in squares of about 1 km with at least 5 cafes. Grey: too few cafes.", SOURCE]))
chart

<br>

`clamp=True` gives anything over £5 the darkest colour, so one very dear square doesn't stretch the scale for all the rest. Hover over the darkest squares: Knightsbridge and South Kensington, both over £5.

Save the chart for your website:

In [ ]:
chart.save("london_latte_map.json")

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1. A pint.** Map the median price of a pint (`canon` values `"lager"`, `"ipa"`, `"guinness"`, ...) in pubs: `vtype == "Pub & bar"`. Copy the cleaning steps from DS8_2.

**2. Bigger squares.** Double `LAT_STEP` and `LNG_STEP`. You lose detail, but keep more of London. Which do you prefer?

**3. Then and now.** The file goes back to 2012. Map the latte in 2019 (`london["year"] == 2019`), with bigger squares. Where did prices rise most?

**4. Cafes or pubs?** Count the venues of each type (`vtype`) in each square, and map where cafes outnumber pubs.

In [ ]:
# Your code here
